<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/03_slurm_scheduling_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖥️ WE7 · Notebook 03 — You are now the cluster scheduler
## Four nodes, four research groups, more work than the cluster can run at once

A cluster is a small number of nodes (computers) shared by a large number of people. Let's say a research institute has
**four** nodes, and four groups want to use them: Biomedical Imaging, Climate Modelling, Robotics and Economics.

This morning all four groups submit jobs at the same time. There are not enough nodes for
everyone, so somebody has to answer questions like these:

- Which job starts first, and why?
- A job needs three nodes and only two are free. Does it wait? Does everyone behind it wait too?
- Two nodes are sitting idle (unused) while jobs are still queueing. How can that happen?
- One group has used the cluster all week. Should that count against them now?

None of these has an obviously correct answer. But by the end of this notebook you will have written a
program that answers all four and you will be able to say who your answers favour.

**How this notebook works**

- Short explanations, then small hands-on tasks marked **🎯** for you to fill in.
- **Interactive widgets** to play with each idea *before* any arithmetic shows up.
- Every technical word is defined the first time it appears.
- The whole cluster is simulated with plain integers. No GPU is needed and nothing is installed.
- ⏱️ About 70 minutes; each part carries its own estimate. There is more reading and clicking
  than typing, on purpose.

First, three short setup cells. Then we look at the cluster.

## Setup — run these three cells first

This notebook is **self-contained**: the first cell pulls the exercise files (the `sched_viz.py`
display helpers, which also contain the simulator) directly from the course repository. Nothing
to install by hand, no account and no access token needed. Run the setup cells below in order.

In [1]:
#@title ⚙️ Setup 1 of 3 — fetch the exercise files (run me) { display-mode: "form" }
import os, sys, subprocess

REPO_OWNER  = "eth-fdd-fs26"
REPO_NAME   = "FDD-WE7-public"
REPO_BRANCH = "main"
HELPER      = os.path.join("1_slurm_scheduling", "exercise", "sched_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _git(args):
    # Run git without a terminal prompt: the repo is public, so no credentials are needed.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    done = subprocess.run(args, capture_output=True, text=True, env=env)
    return done.returncode, (done.stdout + done.stderr).strip()

if _in_colab():
    url = "https://github.com/%s/%s.git" % (REPO_OWNER, REPO_NAME)

    if os.path.isdir(REPO_NAME):
        print("Updating the exercise repo to the latest version...")
        code, log = _git(["git", "-C", REPO_NAME, "pull", "-q", url, REPO_BRANCH])
    else:
        print("Cloning the exercise repo...")
        code, log = _git(["git", "clone", "-q", "-b", REPO_BRANCH, url, REPO_NAME])

    if code != 0:
        print(log)
        raise RuntimeError(
            "git failed. Check that you are online and run this cell again. If it keeps "
            "failing, delete the %r folder in the file browser (left sidebar) and retry."
            % REPO_NAME)

# Move to the REPO ROOT - the folder holding `1_slurm_scheduling/exercise/` - so imports resolve.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Cloned, but %s is not there - the clone did not bring the exercise files." % HELPER)
sys.path.insert(0, os.path.join(os.getcwd(), "1_slurm_scheduling", "exercise"))
print("Working directory:", os.getcwd())

Cloning the exercise repo...
Working directory: /content/FDD-WE7-public


**Setup 2 of 3 — install dependencies.** Both of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [2]:
#@title 📦 Setup 2 of 3 — install dependencies (run me) { display-mode: "form" }
%pip install -q -r 1_slurm_scheduling/exercise/requirements_slurm.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 638.7/638.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.17.0 which is incompatible.


**Setup 3 of 3 — import the helpers.** The cluster map, the timeline and the quizzes live in
**`sched_viz`** so the teaching cells stay about the *idea* rather than about HTML.

In [3]:
#@title 🧰 Setup 3 of 3 — import the helpers (run me) { display-mode: "form" }
import importlib
import sched_viz as sv
importlib.reload(sv)       # pick up the latest helpers even if a stale copy was cached

print("Cluster ready ✅")
for node in sv.NODE_SPECS:
    print("  {name}: {cores} cores, {mem} GB, {gpus} GPU".format(**node))

Cluster ready ✅
  node01: 8 cores, 64 GB, 0 GPU
  node02: 8 cores, 64 GB, 0 GPU
  node03: 8 cores, 64 GB, 2 GPU
  node04: 8 cores, 64 GB, 2 GPU


---
# Before we start · What are we even looking at?

Two questions the whole hour depends on: **what a node looks like**, and **what
a job needs one for.**

Click each row below.

In [4]:
#@title 🖥️ Click through: what a node is, and what a job asks it for (run me) { display-mode: "form" }
sv.what_is_a_node()

## Why you cannot just start your job

One rule separates a cluster from your laptop, and everything else in this notebook follows from
it.

<div style="border-left:4px solid #764ba2;background:#f7f3ff;border-radius:0 10px 10px 0;padding:15px 19px;margin:16px 0;color:#24262b;font-size:15px;line-height:1.62"><div style="font-size:11px;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:#5b3a80;margin-bottom:8px">The rule everything follows from</div><b>A job owns what it is allocated.</b> The cores, the memory and the GPUs it asked for are handed over when it starts, and stay its own until it finishes. Nobody else can touch them in the meantime.</div>

While your job runs, those cores and that memory belong to you alone, even in the minutes when
your job is barely using them.

### What happens when everyone just starts their own

Suppose everyone could start their jobs themselves. Four people log in at nine in the morning and
each picks a node by hand. All four pick the same one, because nothing stops them. Together they
ask for more memory than that node has, so it starts killing jobs to stay alive. All four lose
their morning's work.

That is why access to a cluster is **handed out**, never taken. You do not start your job. You
**describe** it — how many nodes, how much memory, whether you need a GPU, and for how long — and
then you wait to be given a slot.

### Enter Slurm

<div style="border-left:4px solid #764ba2;background:#f7f3ff;border-radius:0 10px 10px 0;padding:15px 19px;margin:16px 0;color:#24262b;font-size:15px;line-height:1.62"><div style="font-size:11px;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:#5b3a80;margin-bottom:8px">What Slurm is</div><b>Slurm</b> is the program that hands out the slots. It holds every request in a <b>queue</b>, watches what is free, and decides which job gets which nodes, and when.</div>

On a real cluster — ETH's **Euler**, for instance — that queue holds thousands of jobs from
hundreds of people at once, day and night. Nobody negotiates and nobody takes turns by hand. The
program decides.

**How it decides is what you are about to build.**

### Three consequences, and they are the plot of the next hour

Because a job *owns* what it was given, for as long as it declared:

| | |
|---|---|
| 🕳️ | nodes can sit **completely idle (unused)** while jobs are still waiting, because what is free is in the wrong **shape** |
| ⏳ | a **big** job may have to wait for a gap that does not exist yet, and might never appear on its own |
| 🧾 | every **node-hour** can be **charged** to a group, because every allocation is written down |

Idle nodes, waiting users, and a bill. Those are the three things you will spend the next hour
trading against each other.

## 🗺️ The hour ahead

Seven parts. Each one builds on the part before it, so every idea arrives in its turn.

**Parts 0 to 2 — make it work at all**

| part | | what it adds |
|:--:|---|---|
| **0** | *What fits* | one frozen instant: which of these jobs could even start right now? |
| **1** | *By hand* | add a clock. Place five jobs yourself, then measure what you did. |
| **2** | *Automate* | thirty jobs. Write the rule down as code and let it run. |

**Parts 3 to 6 — make it work *well*, and then argue about what "well" means**

| part | | what it adds |
|:--:|---|---|
| **3** | *Wall time* | the one number the user declares — and the two ways to get it wrong. |
| **4** | *Priority* | when first-come-first-served is not good enough, who goes first? |
| **5** | *Backfilling* | fill the gaps in front of a big job without breaking its promise. |
| **6** | *Your policy* | throughput against fairness, your turn to configure it and say who pays. |

You are at **Part 0**. By **Part 6** you will be configuring a real scheduling policy and saying
out loud which group it favours.

---
# Part 0: Look at what you have  ·  ~5 min

No code in this part. Just look, click, and form an opinion.

Below is your cluster as it stands right now. Two jobs are already running, which is why parts of
`node01` and `node03` are in use. Five people have just submitted work.

Three things to notice before you touch anything:

- **A job does not have to take a whole node.** `node01` is running a job that took 4 of its 8
  cores and 32 of its 64 GB; another job could take what is left. A node holds as many jobs as its
  cores, memory and GPUs allow.
- **A GPU job can only go where there are GPUs.** Only `node03` and `node04` have any.
- **Most of the cluster is free.** 22 of the 32 cores are unclaimed, spread over all four nodes.
  Hold on to that number — it is about to stop mattering.

In [5]:
#@title 🗺️ Your cluster, right now (run me) { display-mode: "form" }
sv.cluster_map()

### 🧠 Quick check

In [6]:
#@title 🧠 Quick check (run me) { display-mode: "form" }
sv.mc_quiz("why_scheduler")

### ✅ What you just did

You decided which jobs to start, and in what order. Whatever order you tried, at least two jobs
were left waiting in the queue. Some orders also left nodes idle.

Everything here happened at a single frozen moment, and each job you started kept its nodes for
good. In **Part 1** you will see what changes once every job also has a **duration**: jobs finish,
their nodes become free again, and the order you choose matters far more.

---
# Part 1: Schedule it by hand  ·  ~8 min

Time to pick a rule. Here is the simplest one anybody has ever proposed:

> ### **FIFO** (first in, first out), also called **FCFS** (first come, first served):
> ### consider the jobs in the order they arrived, and start each one as soon as its resources
> ### are free.

No cleverness, no favourites, no negotiation. It is the rule at a bakery counter.

For this first pass we drop cores, memory and GPUs and let every job take **whole nodes**. Packing
in three dimensions by hand is a nightmare; we will hand that part to code in Part 2.

## 1.1 · Five jobs, and the picture to keep in your head

These are the five jobs from the lecture. All five were submitted at the **same moment**, and they
are listed in **priority order**: J1 first.

In [7]:
#@title 🎫 The five jobs from the lecture (run me) { display-mode: "form" }
jobs = sv.five_jobs()

for job in jobs:
    print("{name}: {nodes} node(s) for {requested_time} h   (group: {group})".format(**job))

J1: 1 node(s) for 3 h   (group: bio)
J2: 4 node(s) for 2 h   (group: clim)
J3: 3 node(s) for 4 h   (group: bio)
J4: 3 node(s) for 3 h   (group: robo)
J5: 1 node(s) for 2 h   (group: econ)


Now the mental image that runs through the entire notebook:

In [8]:
#@title 📊 The same five jobs, drawn as rectangles (run me) { display-mode: "form" }
sv.job_shapes()

J2 is short and fat: four nodes, two hours. J3 is tall and long: three nodes, four hours. Scheduling
is packing those rectangles into a grid four rows high, with time running to the right, **without
ever overlapping them**, because a job owns its nodes outright.

## 1.2 · 🕹️ Your turn: place them

Place the five rectangles yourself. The queue enforces FIFO order, so you always place the job at
the head of the queue, but **where** and **when** it goes is your call. Click the job, then click
the cell where its top-left corner belongs.

Try to get the idle count as low as you can. Then press **👀 Show what plain FIFO does** and
compare.

In [9]:
#@title 🕹️ Place the five jobs yourself (run me) { display-mode: "form" }
sv.hand_scheduler()

## 1.3 · 🎯 Four numbers, computed by hand

You now have a schedule. Is it a good one? That question has no single answer, and the four
numbers below are the reason why. Work them out on the **plain FIFO** schedule (the one the
👀 button drew): J1 at 0h, J2 at 3h, J3 at 5h, J4 at 9h, J5 at 9h.

Nothing here is hidden inside a function. Every intermediate value is a variable you can print.

**Waiting time** is how long a job sat in the queue before it started.

In [10]:
submission_time = 0        # every one of the five was submitted at 0h
start_time      = 9        # ...and J4 started at 9h

waiting_time = start_time - submission_time         # 🎯 how long did J4 wait?
print("J4 waited", waiting_time, "h")

J4 waited 9 h


**Turnaround time** is how long the whole thing took from the user's point of view: waiting
*plus* running.

In [11]:
completion_time = 12       # J4 ran for 3 h after starting at 9 h

turnaround_time = waiting_time + completion_time - start_time      # 🎯 from submitting to having the results
print("J4's turnaround was", turnaround_time, "h  (of which", waiting_time, "h was queueing)")

J4's turnaround was 12 h  (of which 9 h was queueing)


**Makespan** is the one you have probably never met, so here it is spelled out:

> ### **Makespan** = the total wall-clock span of the whole workload, from the first job starting
> ### to the last job finishing.

It is not about any single job. It answers "how long did this pile of work occupy the cluster?".

In [12]:
first_start     = 0        # J1 went first
last_completion = 12       # J4 finished last

makespan = last_completion - first_start
print("makespan:", makespan, "h")

makespan: 12 h


**Utilisation** is how much of the machine actually did work while all that was going on.

> ### **Utilisation** = busy node-hours ÷ node-hours available over the makespan.

A *node-hour* is one node kept busy for one hour. Four nodes for twelve hours is 48 node-hours of
capacity; whatever the jobs did not use was paid for and thrown away.

In [15]:
busy_node_hours = 1*3 + 4*2 + 3*4 + 3*3 + 1*2      # nodes × hours, one term per job
total_node_hours = 4 * makespan                             # 🎯 four nodes, for the whole makespan

utilisation = busy_node_hours / total_node_hours
idle_node_hours = total_node_hours - busy_node_hours

print("busy :", busy_node_hours, "node-hours")
print("total:", total_node_hours, "node-hours")
print("idle :", idle_node_hours, "node-hours  ← paid for, never used")
print("utilisation: {:.1%}".format(utilisation))

busy : 34 node-hours
total: 48 node-hours
idle : 14 node-hours  ← paid for, never used
utilisation: 70.8%


## 1.4 · Two audiences, four numbers

Split those four into two columns, because they are answering different people's questions:

| number | question it answers | who is asking |
|---|---|---|
| waiting time | how long did **I** sit in the queue? | the researcher |
| turnaround time | how long until **I** had my results? | the researcher |
| makespan | how long did the whole pile take? | you, running the cluster |
| utilisation | how much of the machine did work? | you, and whoever paid for the nodes |

Nothing forces those two columns to agree. A schedule can be excellent for the cluster and
miserable for one user, and the other way round. **No single number tells you whether a schedule
was good**, a sentence worth remembering when a vendor shows you one.

### 🧠 Quick check

In [16]:
#@title 🧠 Quick check: the four numbers (run me) { display-mode: "form" }
sv.number_quiz("metrics_math")

In [17]:
#@title 🧠 Quick check: true or false (run me) { display-mode: "form" }
sv.true_false_quiz("metrics")

> ### What you just did
> The simplest possible rule, applied honestly, produced **14 idle node-hours out of 48** and left
> one user waiting nine hours. Nobody made a mistake. FIFO did exactly what FIFO does. Hold on to
> that hole between 0h and 3h (three nodes, three hours, wide open), because Part 5 is about
> filling it.

---
# Part 2: Write the rule down as code  ·  ~10 min

Five jobs was a game. Here is a Monday morning with **thirty**, which is still a very small
cluster day.

In [18]:
#@title 🎫 Thirty jobs, this time asking for parts of nodes (run me) { display-mode: "form" }
jobs30 = sv.workload(n=30, seed=4)          # all submitted at 0h, all wall times honest

for job in jobs30[:6]:
    print("{name}: {nodes} node(s) × {cores} cores, {mem} GB, {gpus} GPU, "
          "{requested_time} h   ({group})".format(**job))
print("...")
print(len(jobs30), "jobs in the queue")

j01: 1 node(s) × 8 cores, 64 GB, 0 GPU, 1 h   (bio)
j02: 2 node(s) × 4 cores, 32 GB, 0 GPU, 6 h   (bio)
j03: 2 node(s) × 4 cores, 32 GB, 0 GPU, 2 h   (bio)
j04: 1 node(s) × 4 cores, 32 GB, 0 GPU, 2 h   (bio)
j05: 2 node(s) × 4 cores, 32 GB, 1 GPU, 3 h   (bio)
j06: 2 node(s) × 4 cores, 32 GB, 1 GPU, 3 h   (robo)
...
30 jobs in the queue


Placing those by hand is over, and notice why: these jobs ask for **parts** of nodes. `j03`
might want 2 cores and 16 GB, which leaves room for two more jobs on the same node. That is the
packing we skipped in Part 1, and it is exactly the kind of bookkeeping a program is good at and
you are not.

So let us write the rule down. A scheduling policy is nothing more than an algorithm, and it has
five moving parts. We will build them one at a time, printing the state after each.

## 2.1 · The cluster, as data

Four dictionaries. `free_cores`, `free_mem` and `free_gpus` are what is left; they go down when a
job starts and back up when it ends. That is the entire state of the machine.

In [19]:
#@title 🗄️ The cluster, as four dictionaries (run me) { display-mode: "form" }
cluster = sv.new_cluster()

for node in cluster:
    print(node)

{'name': 'node01', 'cores': 8, 'mem': 64, 'gpus': 0, 'free_cores': 8, 'free_mem': 64, 'free_gpus': 0}
{'name': 'node02', 'cores': 8, 'mem': 64, 'gpus': 0, 'free_cores': 8, 'free_mem': 64, 'free_gpus': 0}
{'name': 'node03', 'cores': 8, 'mem': 64, 'gpus': 2, 'free_cores': 8, 'free_mem': 64, 'free_gpus': 2}
{'name': 'node04', 'cores': 8, 'mem': 64, 'gpus': 2, 'free_cores': 8, 'free_mem': 64, 'free_gpus': 2}


## 2.2 · 🎯 Step 1: pick the next job

FIFO means the queue is a plain list and you take from the **front**.

<details><summary>💡 <b>Hint 1</b>: what are we actually asking?</summary>

A Python list has one method that removes an item *and* hands it back to you. FIFO takes from the front of the queue, not the back, and lists are indexed from the front.

</details>

<details><summary>🔧 <b>Hint 2</b>: which variable, which function</summary>

`queue.pop(i)` removes the item at position `i` and returns it. Which `i` is the front?

</details>

In [20]:
queue = list(jobs30)          # a copy we are allowed to consume

next_job = queue.pop(0)                # 🎯 take the job at the front of the queue
print("next up:", next_job["name"], "-", next_job["nodes"], "node(s) ×",
      next_job["cores"], "cores,", next_job["requested_time"], "h")
print(len(queue), "left in the queue")

next up: j01 - 1 node(s) × 8 cores, 1 h
29 left in the queue


## 2.3 · 🎯 Step 2: find nodes with enough free resources

A node can take the job only if **all three** of its free resources cover what the job asked for.
Then we need `next_job["nodes"]` such nodes.

<details><summary>💡 <b>Hint 1</b>: what are we actually asking?</summary>

A node is a candidate when its free cores cover the request, *and* its free memory does, *and* its free GPUs do. Collect the candidates first, then take as many as the job asked for.

</details>

<details><summary>🔧 <b>Hint 2</b>: which variable, which function</summary>

The job's three requests are `next_job["cores"]`, `next_job["mem"]` and `next_job["gpus"]`. The node's three leftovers are `node["free_cores"]`, `node["free_mem"]` and `node["free_gpus"]`.

</details>

In [21]:
candidates = []
for node in cluster:
    if (node["free_cores"] >= next_job["cores"] and
            node["free_mem"] >= next_job["mem"] and              # 🎯 memory
            node["free_gpus"] >= next_job["gpus"]):               # 🎯 GPUs
        candidates.append(node["name"])

chosen = candidates[:next_job["nodes"]]              # take as many as the job asked for
print("candidates:", candidates)
print("chosen    :", chosen)

candidates: ['node01', 'node02', 'node03', 'node04']
chosen    : ['node01']


> 🔧 **From here on, `sv.find_nodes(cluster, job)` does exactly what you just wrote**, and
> returns `None` when there are not enough candidates. It adds one refinement: it tries the
> GPU-free nodes first, so a plain CPU job does not squat on a GPU node that a GPU job will need in
> ten minutes. Try it and check you get the same answer.

In [22]:
print(sv.find_nodes(cluster, next_job))

['node01']


## 2.4 · 🎯 Step 3: allocate

Allocating is subtraction. Nothing more mysterious than that.

<details><summary>💡 <b>Hint 1</b>: what are we actually asking?</summary>

The job takes its cores, memory and GPUs out of *each* node it was given, so what is left on that node goes down by exactly the amount requested.

</details>

<details><summary>🔧 <b>Hint 2</b>: which variable, which function</summary>

You want `-=` on `node["free_cores"]`, `node["free_mem"]` and `node["free_gpus"]`.

</details>

In [23]:
print("before:", cluster[0])

for node in cluster:
    if node["name"] in chosen:
        node["free_cores"] -= next_job["cores"]
        node["free_mem"]   -= next_job["mem"]                    # 🎯
        node["free_gpus"]  -= next_job["gpus"]                    # 🎯

print("after :", cluster[0])

before: {'name': 'node01', 'cores': 8, 'mem': 64, 'gpus': 0, 'free_cores': 8, 'free_mem': 64, 'free_gpus': 0}
after : {'name': 'node01', 'cores': 8, 'mem': 64, 'gpus': 0, 'free_cores': 0, 'free_mem': 0, 'free_gpus': 0}


## 2.5 · 🎯 Step 4: record the start time

The job is now running. Two numbers describe it: when it started, and when it will be over.

In [24]:
now = 0

start_time = now
end_time   = start_time + next_job["requested_time"]                                     # 🎯 when will this job be finished?

print(next_job["name"], "runs from", start_time, "h to", end_time, "h on", chosen)

j01 runs from 0 h to 1 h on ['node01']


## 2.6 · Step 5: release at completion

The mirror image of Step 3: at `end_time` the resources go back into the pool, and the next job can
have them.

In [25]:
for node in cluster:
    if node["name"] in chosen:
        node["free_cores"] += next_job["cores"]
        node["free_mem"]   += next_job["mem"]
        node["free_gpus"]  += next_job["gpus"]

print("back to full:", cluster[0])

back to full: {'name': 'node01', 'cores': 8, 'mem': 64, 'gpus': 0, 'free_cores': 8, 'free_mem': 64, 'free_gpus': 0}


## 2.7 · 🎯 Put the five steps in a loop

Below is your scheduler. Every line in it is one of the five things you just did, wrapped in a loop
that moves the clock forward to the next moment when anything can change.

**One line is missing, and it is the one that carries the policy.** When the job at the head of the
queue does not fit, what should happen to the jobs behind it?

<details><summary>💡 <b>Hint 1</b>: what are we actually asking?</summary>

FIFO means first come, first served: *served*, not merely considered. If a later job were allowed to jump in front of a blocked one, the rule would no longer be FIFO. So what has to happen to the loop over the queue?

</details>

<details><summary>🔧 <b>Hint 2</b>: which variable, which function</summary>

You are inside `while queue:`. Python has two keywords for leaving a loop early: one abandons it entirely, the other skips to the next item. Which one refuses to look any further down the queue?

</details>

In [26]:
def fifo_schedule(jobs):
    cluster  = sv.new_cluster()
    queue    = list(jobs)
    running  = []          # (job, nodes, end_time) for everything currently on the machine
    schedule = []          # what we will draw at the end
    now      = 0

    while queue or running:
        # --- step 5: give back the resources of everything that has finished
        for job, nodes, end_time in list(running):
            if end_time <= now:
                for node in cluster:
                    if node["name"] in nodes:
                        node["free_cores"] += job["cores"]
                        node["free_mem"]   += job["mem"]
                        node["free_gpus"]  += job["gpus"]
                running.remove((job, nodes, end_time))

        # --- steps 1 to 4: start whatever fits, in queue order
        while queue:
            job    = queue[0]                       # step 1: the head of the queue
            chosen = sv.find_nodes(cluster, job)    # step 2: where could it go?
            if chosen is None:
                break                                 # 🎯 the head does not fit. Now what?
            queue.pop(0)
            for node in cluster:                    # step 3: allocate
                if node["name"] in chosen:
                    node["free_cores"] -= job["cores"]
                    node["free_mem"]   -= job["mem"]
                    node["free_gpus"]  -= job["gpus"]
            start_time = now                        # step 4: record
            end_time   = start_time + job["requested_time"]
            running.append((job, chosen, end_time))
            schedule.append({"name": job["name"], "group": job["group"], "nodes": chosen,
                             "n_nodes": job["nodes"], "cores": job["cores"], "gpus": job["gpus"],
                             "start": start_time, "end": end_time, "submit_time": 0,
                             "requested_time": job["requested_time"],
                             "actual_duration": job["requested_time"], "killed": False,
                             "backfilled": False, "waiting": start_time,
                             "turnaround": end_time})

        # --- move the clock to the next moment anything can change
        if running:
            now = min(end_time for _, _, end_time in running)
        else:
            break

    return schedule


my_schedule = fifo_schedule(jobs30)
print(len(my_schedule), "jobs placed")

30 jobs placed


**Draw it.** `sv.timeline` takes a schedule and gives you the picture from Part 1, at
whatever size the workload happens to be.

In [27]:
#@title 📊 Draw your thirty-job schedule, and measure it (run me) { display-mode: "form" }
sv.timeline(my_schedule, px=30, title="Thirty jobs, FIFO",
            note="Each node row is eight core-slots tall, so a node can hold several jobs at "
                 "once: that is the partial-node packing we skipped in Part 1. Hover any "
                 "rectangle for its nodes, cores and hours.")
sv.metric_strip(sv.metrics(my_schedule),
                keys=["makespan", "utilisation", "idle_node_hours", "avg_waiting", "avg_turnaround"])

**Check yourself against the reference.** `sv.run_schedule` is the same loop you just wrote,
with the extra options we will need later. On FIFO it should agree with you exactly.

In [28]:
#@title ✅ Check your loop against the reference implementation (run me) { display-mode: "form" }
reference = sv.run_schedule(jobs30)          # no priorities, no backfilling: plain FIFO

print("yours    :", [(s["name"], s["start"]) for s in my_schedule[:5]])
print("reference:", [(s["name"], s["start"]) for s in reference[:5]])
print("same makespan?", sv.metrics(my_schedule)["makespan"] == sv.metrics(reference)["makespan"])

yours    : [('j01', 0), ('j02', 0), ('j03', 0), ('j04', 0), ('j05', 2)]
reference: [('j01', 0), ('j02', 0), ('j03', 0), ('j04', 0), ('j05', 2)]
same makespan? True


> ### What you just did
> You wrote a scheduler. Not a toy version of one. The loop above is the shape of the real thing:
> release, order the queue, fit what fits, advance the clock. Everything for the rest of the hour
> is a change to **one** of those four lines. Backfilling changes what happens when the head does
> not fit. Priority changes the order of the queue. That is all.

---
# Part 3: The number you have to declare  ·  ~8 min

Go back one cell and look at the line you wrote in Step 4:

```python
end_time = start_time + job["requested_time"]
```

Your simulator moved the clock forward by three hours for J1. **How did it know that J1 lasts three
hours?**

It did not measure anything. It did not profile the code, look at the input file, or remember the
last time this user ran something similar. It read a number out of the job's dictionary, and that
number got there because **the user typed it in the submission script**. In Slurm it is one line:

```bash
#SBATCH --time=03:00:00
```

> ### Slurm never knows how long a job will take. It only knows the **limit that was requested**.

That number has a name, and you will see it everywhere on a cluster:

> ### **Wall time** = elapsed clock time from a job starting to it stopping. Not CPU time, not
> ### compute time, but the time you would measure with a clock on the wall. `--time` declares the
> ### **maximum** wall time the job is allowed, and the job is killed the moment it reaches it.

That is not a simplification for teaching. It is how the thing works, and everything in Part 5
depends on it.

## 3.1 · Two fields from here on

From this point every job carries two durations, and they play completely different roles:

| field | what it is | who may read it |
|---|---|---|
| `requested_time` | the number the user declared, `--time` | **the scheduler**, and nothing else |
| `actual_duration` | how long the work really takes | only the simulation, to move the clock |

`actual_duration` is a fiction of this notebook. On a real cluster nobody has it: it does not exist
until the job has already ended. Whenever you see the scheduler make a decision below, check which
of the two it used.

In [29]:
jobs = sv.five_jobs()
j4 = jobs[3]

print("J4 declared :", j4["requested_time"], "h   ← the scheduler plans on this")
print("J4 really   :", j4["actual_duration"], "h   ← only the simulation clock knows this")
print()
print("Honest so far. Now change one of them and nothing about the work changes:")
j4["requested_time"] = 6
print("J4 declared :", j4["requested_time"], "h")
print("J4 really   :", j4["actual_duration"], "h")

J4 declared : 3 h   ← the scheduler plans on this
J4 really   : 3 h   ← only the simulation clock knows this

Honest so far. Now change one of them and nothing about the work changes:
J4 declared : 6 h
J4 really   : 3 h


## 3.2 · 🕹️ One slider, two ways to get it wrong

Below is a single job: three nodes, and **three hours of real work that never changes**. The only
thing you can move is `--time`.

Push it below three hours, then well above, and read the three counters underneath each time.

In [30]:
#@title 🕹️ One slider: declare too little, or too much (run me) { display-mode: "form" }
sv.walltime_slider()

### Under-declare, and the job is killed

At the limit Slurm sends the job a signal and then kills it. Not "warns", not "extends": kills. And
the node-hours it burnt up to that moment are **still charged to your group**. Unless the job wrote
checkpoints to disk, everything it computed is gone.

Here is that happening in a workload rather than in a widget. Two of these thirty users got their
wall time wrong.

In [31]:
#@title 💀 Who gets killed at their declared limit, and what it cost (run me) { display-mode: "form" }
jobs30_real = sv.workload(n=30, seed=4, honest=False)     # same shapes, honest wall times off

for job in jobs30_real:
    if job["actual_duration"] > job["requested_time"]:
        wasted = job["nodes"] * (job["cores"] / 8) * job["requested_time"]
        print("{name} ({group}): declared {requested_time} h, needs {actual_duration} h "
              "→ killed at the limit, {w:.1f} node-hours burnt for nothing"
              .format(w=wasted, **job))

j15 (bio): declared 2 h, needs 3 h → killed at the limit, 2.0 node-hours burnt for nothing
j24 (econ): declared 3 h, needs 5 h → killed at the limit, 1.5 node-hours burnt for nothing


In [32]:
#@title 📊 The same day, with dishonest wall times (run me) { display-mode: "form" }
schedule_real = sv.run_schedule(jobs30_real, backfill=True)
sv.timeline(schedule_real, px=30,
            title="A red outline marks a job that was killed at its declared limit",
            note="Every red rectangle is a job whose owner asked for less time than the work "
                 "needed. It ran until its declared limit, was killed there, and its group was "
                 "charged for every node-hour it burnt.")

### Over-declare, and nothing is killed, which is exactly why it is dangerous

Pad your wall time and you are safe from the killer, you are billed only for what you use, and
absolutely nothing complains. The cost is invisible and it is real: **the scheduler now believes
your job is a wider rectangle than it is**, and will only ever look for a hole that big.

Look at the picture below. Solid colour is real work. The dashed tail on each rectangle is time
that was declared and never used: empty space every one of these jobs is carrying around.

In [33]:
#@title 📊 Solid = the work. Dashed = declared and never used (run me) { display-mode: "form" }
sv.timeline(schedule_real, px=30, show_declared=True,
            title="Solid = the work. Dashed = declared and never used.",
            note="Nobody is punished for any of that dashed area. Part 5 turns it into a number.")

### 🧠 Quick check

In [34]:
#@title 🧠 Quick check (run me) { display-mode: "form" }
sv.mc_quiz("walltime")

In [35]:
#@title 🧠 Quick check: true or false (run me) { display-mode: "form" }
sv.true_false_quiz("rectangle")

> ### What you just did
> In Slurm a job is a **promise about size and duration**, and the scheduler operates entirely on
> the promise. Both dimensions of your rectangle are declared by you, in advance, and neither is
> ever measured. Declare too little and the job dies. Declare too much and it stops fitting
> anywhere, which is the next thing we are going to measure.

---
# Part 4: Priority, the order is a policy  ·  ~8 min

No real cluster runs FIFO. Bakery-counter order sounds fair until the group that funded half the
hardware is stuck behind a student who submitted two hundred one-minute jobs at eight in the
morning.

Slurm therefore gives every waiting job a **score**, and sorts the queue by that score. The score
is a weighted sum of **factors**, each one a thing the institution has decided should count. A real
site mixes in several — including which **partition** a job was sent to and how big the request is
— but three of them carry the argument, and those three are the ones we will use.

> ### **Job age** = how long the job has already been sitting in the queue. It is the only factor
> ### that grows **on its own**, with nobody deciding anything: a job nobody is asking about
> ### climbs simply because time passes. It is what stops a job from waiting for ever.

> ### **Fair-share** = how much of the cluster the job's **group** has consumed lately, compared
> ### with the share that group was promised. A group that has been over-consuming is pushed down,
> ### a group that has been quiet is pushed up. Section 4.2 is entirely about this one.

> ### **QoS** (quality of service) = a named service class the administrators grant, carrying its
> ### own priority boost **and** its own limits. You pick one on submission with `--qos=...`, and
> ### you cannot pick one you were not granted.

For example, a `debug` QoS might give a large boost but cap the job at 30 minutes on a single node
— so a five-minute test jumps the queue without being able to occupy the machine — while a `long`
QoS might allow a seven-day job but carry a priority **penalty** to pay for it. The boost is
attached to a **class of work**, not to whoever happens to be in a hurry.

Give each of the three a weight and you have a policy. What follows is **not** Slurm's actual
formula, but it has the same shape:

$$\texttt{priority} \;=\; 0.5 \cdot \texttt{age} \;+\; 0.3 \cdot \texttt{fair\_share}
\;+\; 0.2 \cdot \texttt{qos}$$

Each factor is scaled to sit between **0 and 1** before it is weighted: `age` reaches 1 after a
full day of waiting, `qos` is 0.5 for a normal job and 1.0 for a boosted one, and `fair_share` is
the factor $F$ that Section 4.2 builds. A weight is therefore just "how many points this factor
can be worth at most", and **the three weights are the policy** — they are the only thing you get
to set.

## 4.1 · 🕹️ Pick a stance and watch the queue reorder

Six jobs are waiting. Choose a stance with the buttons, or move the weights yourself. Each bar is
split into three segments, one per factor, and a segment is that factor's **value multiplied by
its weight** — its contribution to the score. The whole bar is therefore the score itself, which
is why the longest bar goes first, and hovering over a segment shows you the arithmetic behind
it.

In [36]:
#@title 🕹️ Six jobs, three weights: move them and watch the queue reorder (run me) { display-mode: "form" }
sv.priority_mixer()

### 🎯 Now break it on purpose

Set the **age weight to zero** with the slider above and look at the bottom of the list. Nothing
about those jobs will ever improve.

To see what that costs, we need a workload that behaves like a real day rather than a snapshot:
one **wide** job, `W` (three nodes for six hours, from the group that has been consuming most),
and a stream of **narrow** jobs from the other groups arriving three per hour, all day, each one
easy to fit.

In [37]:
#@title 🎫 One wide job, and a stream of narrow ones that never stops (run me) { display-mode: "form" }
stream = sv.starving_workload()          # W, plus 72 small jobs arriving over 24 h

print(len(stream), "jobs")
print("the wide one:", {k: stream[0][k] for k in ["name", "nodes", "requested_time", "group", "qos"]})
print("a narrow one:", {k: stream[5][k] for k in ["name", "nodes", "requested_time", "group", "qos"]})

73 jobs
the wide one: {'name': 'W', 'nodes': 3, 'requested_time': 6, 'group': 'bio', 'qos': 1.0}
a narrow one: {'name': 's05', 'nodes': 1, 'requested_time': 2, 'group': 'robo', 'qos': 2.0}


In [40]:
no_age   = {"age": 0.0, "fair_share": 0.3, "qos": 0.7}   # 🎯 switch ageing off entirely
with_age = {"age": 0.5, "fair_share": 0.3, "qos": 0.2}   # ...and a policy that keeps it, to compare against

# The next cell schedules the very same day twice, once with each of these two sets of weights.

In [41]:
#@title ▶️ Schedule the same day twice, once with each set of weights (run me) { display-mode: "form" }
for label, weights in [("age weight 0.0", no_age), ("age weight 0.5", with_age)]:
    sched = sv.run_schedule(stream, backfill=True, weights=weights, dynamic_fairshare=True)
    W = [s for s in sched if s["name"] == "W"][0]
    print("%-16s  W started at %2d h, having waited %2d h   |   everyone else waited %.2f h "
          "on average" % (label, W["start"], W["waiting"], sv.metrics(sched)["avg_waiting"]))
    print("%-16s  W scored %.3f the moment it was submitted, and %.3f when it started\n"
          % ("", sv.score_at(stream, sched, "W", 0, weights),
             sv.score_at(stream, sched, "W", W["start"], weights)))

age weight 0.0    W started at 26 h, having waited 26 h   |   everyone else waited 1.07 h on average
                  W scored 0.650 the moment it was submitted, and 0.650 when it started

age weight 0.5    W started at  1 h, having waited  1 h   |   everyone else waited 4.49 h on average
                  W scored 0.400 the moment it was submitted, and 0.421 when it started



**That is starvation.** With the age weight at zero, a job's score on the day it arrives is
its score for ever — and the second line of that output says it outright: `W` scored **0.650** the
moment it was submitted, and **0.650** twenty-six hours later, when it finally started. Not one
thing about the job improved while it waited. It never reaches the front, because a fresh narrow
job with a better QoS outranks it every single round, and fresh narrow jobs keep arriving. Nothing
is rejected and nothing is broken. `W` simply waits until the stream stops: a full day.

Turn ageing back on and those two numbers stop being equal: `W`'s score climbs while it sits there,
and that slow climb is all it takes. It reaches the front within an hour and gets a reservation.
Without ageing, `W` ran only once the competition dried up; with it, `W` starts an hour after
asking, at a time somebody can plan around. That is what the age term is for — it turns
"eventually, when nobody else wants the machine" into a **timely** start, and it is why every
production scheduler has one.

Now read the end of the first line, because it is the honest half of the story: **the average wait
went up by more than three hours.** Letting the starving job through means seventy-two other jobs
go behind it. And that is also why a real Slurm policy scores the **resource characteristics** of
the request itself, the factor we set aside above: three nodes for six hours is not the same
proposition as one node for one hour, and a site that does not say so in its score ends up saying
it by accident, in everybody else's waiting time.

## 4.2 · Fair-share is a **loop**, not a number

Here is where you, the administrator, actually make policy. You assign each group a **target
share** of the cluster, not because of any mathematics, but because of who funded the hardware and
what the institution promised:

| group | target share | why |
|---|---|---|
| Biomedical Imaging | **40 %** | bought two of the four nodes |
| Climate Modelling | **30 %** | brought the grant that pays for the room |
| Robotics | **20 %** | subscription, joined in year two |
| Economics | **10 %** | subscription, smallest contribution |

The whole of fair-share is built out of two numbers per group, and it is worth being exact about
which is which.

> ### **S**, the **target share** = the fraction of the cluster a group was **promised**: 0.40 for
> ### Biomedical Imaging, 0.10 for Economics. It is the column you just read. You choose it, and it
> ### does not move until you change it.

> ### **U**, the **usage** = the fraction of everything the cluster actually **delivered** recently
> ### that went to this group. It is measured from the accounting records, never chosen, and it
> ### moves every time one of that group's jobs finishes.

So the ratio $U/S$ is a single number for "how far is this group from its promise": **1** means it
got exactly what it was promised, **above 1** means it took more than its share, **below 1** means
it took less. Slurm turns that ratio into the **fair-share factor** $F$:

$$F \;=\; 2^{\,-U/S}$$

Read that off the three cases and it explains itself:

| situation | U / S | F | effect on the group's jobs |
|---|---|---|---|
| on target | 1 | **0.5** | equilibrium, no push either way |
| above target | > 1 | towards 0 | its jobs lose ties |
| below target | < 1 | towards 1 | its jobs win ties |

> ### **Fair-share** does not mean everyone gets the same. It means each group's consumption is
> ### pushed back towards **the share it was assigned**, and the push gets stronger the further
> ### away it drifts.

And it is a **loop**: every finished job is charged to its group, U moves, F moves, the queue
reorders, different jobs run, and next week the charge is different again.

Here is $F$ worked out for one week of that loop, with `U` read off the accounting records:

In [42]:
#@title 🔢 F = 2**(−U/S), group by group, for one week's usage (run me) { display-mode: "form" }
usage = {"bio": 0.62, "clim": 0.25, "robo": 0.10, "econ": 0.03}   # consumed lately

for group in sv.GROUPS:
    S = sv.TARGET_SHARE[group]
    U = usage[group]
    F = 2 ** (-U / S)
    print("%-20s target %.0f%%   consumed %4.0f%%   F = 2**(-%.2f/%.2f) = %.3f"
          % (sv.GROUP_NAME[group], 100*S, 100*U, U, S, F))

Biomedical Imaging   target 40%   consumed   62%   F = 2**(-0.62/0.40) = 0.342
Climate Modelling    target 30%   consumed   25%   F = 2**(-0.25/0.30) = 0.561
Robotics             target 20%   consumed   10%   F = 2**(-0.10/0.20) = 0.707
Economics            target 10%   consumed    3%   F = 2**(-0.03/0.10) = 0.812


### 🕹️ Run the loop, and switch a group off

Below, every week the groups ask for work in proportion to the share they were given, the scheduler
serves them in proportion to F, the node-hours are charged, and F moves.

One control in that widget has not been explained yet, and it is the one that decides how long the
cluster's memory is:

> ### **Half-life** = how long it takes for consumption that has already been charged to count only
> ### **half** as much. Old node-hours are never deleted, they are discounted as they age: with a
> ### half-life of two weeks, node-hours burned two weeks ago weigh 0.5, four weeks ago 0.25, six
> ### weeks ago 0.125 — shrinking for ever, never quite reaching zero.

So **U** is not last week's usage. It is every week's usage, with the older weeks faded out.

Three things to do:

1. Press **▶ Run one week** a few times and watch every curve converge on 0.5. That is the
   equilibrium: everybody consuming exactly their target.
2. Switch **Economics** to **Quiet**, run four weeks, then switch it back to Active. Watch its
   factor climb while it is away, and watch what happens when it returns.
3. Now set the **half-life** slider to **1 week**, press **↺ Back to week 0**, and do step 2 again.
   Then try it with the slider at **8 weeks**. Same groups, same behaviour, two very different
   amounts of forgiveness.

In [43]:
#@title 🕹️ Fair-share, week by week (run me) { display-mode: "form" }
sv.fairshare_loop()

group,target S,consumed U,F = 2−U/S,this week,submitting?


> ⚠️ **Nothing is ever reset.** A group that goes quiet does not get its record wiped at the
> end of the month. Its old consumption is multiplied by one half every half-life, then again, then
> again, until it is too small to matter. The half-life is a policy dial: short and the cluster
> forgets last week's marathon almost immediately, long and a group pays for it for months.

### 🧠 Quick check

In [44]:
#@title 🧠 Quick check (run me) { display-mode: "form" }
sv.mc_quiz("fairshare")

> ### What you just did
> You chose weights, and in doing so you chose whose work matters. **No priority rule is
> universally correct.** Those three numbers encode who the institution wants to protect, not
> anything that could be derived. The only honest way to defend them is to say out loud who pays
> for them, which is exactly what Part 6 asks you to do.

---
# Part 5: Reservation and backfilling  ·  ~10 min

Back to the five jobs, because this is the part the lecture drew on the board.

## 5.1 · Strict priority first, so the numbers land

Consider the queue in priority order and start a job the moment its resources are free. When the
job at the head does not fit, **nobody behind it may start**: the FIFO rule of Part 2, now applied
to a priority order.

In [45]:
jobs = sv.five_jobs()
strict = sv.run_schedule(jobs, backfill=False)     # backfilling off: nobody overtakes the head

In [46]:
#@title 📊 Draw it, and measure it (run me) { display-mode: "form" }
sv.timeline(strict, shade_idle=True, title="Strict priority: the pink is nobody's",
            note="J2 needs four nodes and only three are free, so from 0h to 3h three nodes wait. "
                 "Later, J4 needs three nodes and only one is free, so J5 waits behind it too.")
sv.metric_strip(sv.metrics(strict),
                keys=["makespan", "utilisation", "idle_node_hours", "avg_waiting", "avg_turnaround"])

**12 hours, 14 idle node-hours, 70.8 % utilisation.** Nearly a third of the machine did
nothing, and the jobs that could have used it were sitting in the queue the whole time.

## 5.2 · Two definitions, and they are the same mechanism

> ### **Reservation**: the earliest start time the scheduler can *guarantee* a blocked job, worked
> ### out from what is currently running and written down. No later decision is allowed to push
> ### that job past it.

> ### **Backfilling**: starting a lower-priority job ahead of its turn, but **only** if it is
> ### declared to finish before the reservation, so the reserved job is untouched.

Neither works without the other. Without a reservation, filling holes would be reckless: a wide job
could be overtaken by a stream of narrow ones for ever and never start at all. **The promise made
to the big job is what makes filling the holes in front of it safe.**

## 5.3 · 🕹️ Step through it

Ten steps, exactly the way the slides do it. Click through slowly.

In [47]:
#@title 🕹️ Step through the ten moves, the way the slides do (run me) { display-mode: "form" }
sv.backfill_walkthrough()

## 5.4 · 🎯 Apply the test yourself

The test is two lines, and you have everything you need for both.

<details><summary>💡 <b>Hint 1</b>: what are we actually asking?</summary>

A candidate may start now if it will be finished by the time the reserved job is due. Its finish time is simply now plus how long it is going to run for, as far as the scheduler is concerned.

</details>

<details><summary>🔧 <b>Hint 2</b>: which variable, which function</summary>

It is `current_time + job["..."]`, and there are two duration fields to choose between. Only one of them is a number the scheduler is allowed to look at.

</details>

In [48]:
jobs = sv.five_jobs()
J3, J4, J5 = jobs[2], jobs[3], jobs[4]

current_time   = 0        # J1 has just started
reserved_start = 3        # ...and Slurm has promised J2 all four nodes at 3h

for job in [J3, J4, J5]:
    finish_time  = current_time + job["requested_time"]  # 🎯 when would it be done?
    fits_in_time = finish_time <= reserved_start
    fits_in_size = job["nodes"] <= 3                # three nodes are free right now
    print("%s: declared %d h → finish_time = %d, before the reservation? %-5s | "
          "nodes ok? %-5s | backfill? %s"
          % (job["name"], job["requested_time"], finish_time, fits_in_time, fits_in_size,
             "YES" if (fits_in_time and fits_in_size) else "no"))

J3: declared 4 h → finish_time = 4, before the reservation? False | nodes ok? True  | backfill? no
J4: declared 3 h → finish_time = 3, before the reservation? True  | nodes ok? True  | backfill? YES
J5: declared 2 h → finish_time = 2, before the reservation? True  | nodes ok? True  | backfill? YES


> ### 🔍 Look hard at which field you used.
> `job["requested_time"]`. **Not** `job["actual_duration"]`. The scheduler cannot see the real
> duration and never will. Whether your job gets backfilled is decided entirely by the number you
> typed. That is the payoff of Part 3, and it is the one habit worth taking out of this hour.

## 5.5 · The result

In [49]:
backfilled = sv.run_schedule(jobs, backfill=True)      # one flag, and that is the whole change

In [50]:
#@title 📊 Draw it, and measure it (run me) { display-mode: "form" }
sv.timeline(backfilled, shade_idle=True, title="With backfilling: ⤴ marks the job that jumped",
            note="J4 ran at 0h instead of 9h, on nodes that were otherwise going to stand empty.")
sv.metric_strip(sv.metrics(backfilled),
                keys=["makespan", "utilisation", "idle_node_hours", "avg_waiting", "avg_turnaround"])

**9 hours, 2 idle node-hours, 94.4 % utilisation.** Same cluster, same five jobs, same real
work. Two things have to be said out loud, because they are easy to miss:

- **J2 starts at 3h in both schedules.** It was promised 3h and it got 3h. The entire gain
  (three hours of makespan, twelve node-hours recovered) cost the protected job **nothing**.
- **J3 is the one that moved.** J4 overtook it despite having lower priority. Backfilling does not
  delay the reserved job, but it does reorder everybody else, and J3's user is entitled to notice.

## 5.6 · 🎯 The closing experiment: change one number and nothing else

J4's user gets nervous. The job really takes three hours; they declare **five**, just to be safe.
Not a single line of their code changes.

In [51]:
jobs = sv.five_jobs()
jobs[3]["requested_time"] = 5                 # 🎯 J4 pads its wall time from 3 h to 5 h
print("J4 declares", jobs[3]["requested_time"], "h and really runs", jobs[3]["actual_duration"], "h")

J4 declares 5 h and really runs 3 h


In [52]:
#@title 📊 Draw it, and measure it (run me) { display-mode: "form" }
padded = sv.run_schedule(jobs, backfill=True)

sv.timeline(padded, shade_idle=True, title="Same work, one number changed")
sv.metric_strip(sv.metrics(padded),
                keys=["makespan", "utilisation", "idle_node_hours", "avg_waiting", "avg_turnaround"])

**Back to 12 hours and 70.8 %.** Every gain from backfilling is gone.

`finish_time = 0 + 5 = 5`, and `5 <= 3` is false, so J4 fails the test and waits until 9h, where
it still runs for exactly three hours, because the real work never changed. The machine is the
same, the job is the same, the user is the same. One number is to blame.

> ### 🎯 **The habit to leave with:** ask for a **realistic** wall time. That single number decides
> ### whether your job ever fits in a hole, and holes are where short jobs get to start early.

### 🧠 Quick check

In [53]:
#@title 🧠 Quick check: reservations (run me) { display-mode: "form" }
sv.mc_quiz("reservation")

In [54]:
#@title 🧠 Quick check: which field did it use? (run me) { display-mode: "form" }
sv.mc_quiz("declared_or_real")

In [55]:
#@title 🧠 Quick check: the backfilling arithmetic (run me) { display-mode: "form" }
sv.number_quiz("backfill_math")

In [56]:
#@title 🧠 Quick check: true or false (run me) { display-mode: "form" }
sv.true_false_quiz("backfilling")

---
# Part 6: Design your scheduler  ·  ~8 min

Every setting so far was chosen for you: FIFO in Part 2, wall time in Part 3, one priority
formula in Part 4, one reservation in Part 5. This part hands you all six knobs at once, on a
week you have not seen, so you can feel what they trade against each other.

The trade you will feel the most is **throughput against fairness**. Weight priority towards QoS
and age, and the cluster clears more work, faster. Weight it towards fair-share instead, and the
four groups land closer to the shares they were promised, at the cost of speed. `reserve_depth`
is part of the same trade: Part 5 protected one blocked job with a reservation; here, more than
one can hold a protected place at once.

It is a new week on the same cluster. You pick the policy, then look at what it did to each
group.

**Where these forty jobs come from.** `sv.workload` is the same random job generator behind
every workload in this notebook, not a log from a real cluster. Give it a seed and it invents a
week: forty jobs, arrival times spread across eighteen hours, sizes and groups drawn from the same
odds every time that seed is used. It exists so this notebook can hand you a new week whenever it
needs one, without recording one by hand.

In [57]:
#@title 🎫 Next week: forty jobs arriving over eighteen hours (run me) { display-mode: "form" }
new_week = sv.workload(n=40, seed=31, arrivals=18, honest=False)

print(len(new_week), "jobs, arriving between 0 h and %d h"
      % max(j["submit_time"] for j in new_week))
print("  wide jobs (3 nodes):", sum(1 for j in new_week if j["nodes"] == 3))
print("  jobs needing a GPU :", sum(1 for j in new_week if j["gpus"] > 0))
for g in sv.GROUPS:
    print("  %-20s %2d jobs   (promised %.0f%% of the cluster)"
          % (sv.GROUP_NAME[g], sum(1 for j in new_week if j["group"] == g),
             100 * sv.TARGET_SHARE[g]))

40 jobs, arriving between 0 h and 17 h
  wide jobs (3 nodes): 7
  jobs needing a GPU : 6
  Biomedical Imaging    9 jobs   (promised 40% of the cluster)
  Climate Modelling    13 jobs   (promised 30% of the cluster)
  Robotics             15 jobs   (promised 20% of the cluster)
  Economics             3 jobs   (promised 10% of the cluster)


## 6.1 · What you can turn

Six settings. The three weights are Part 4's. `backfill` and `reserve_depth` are Part 5's
reservation, now able to protect more than one job at a time. `fair_share_live` is Part 4's loop,
recomputed as this week goes on. You pass them to `sv.evaluate` as one dictionary, and **anything
you leave out keeps its default**.

| setting | default | turning it up means |
|---|---|---|
| `w_age` | 0.5 | a job climbs the queue faster the longer it has waited |
| `w_fair_share` | 0.3 | a group that has used more than its promised share gets pushed down |
| `w_qos` | 0.2 | a job with a granted priority class counts for more |
| `backfill` | `True` | small jobs may fill the gaps in front of a reserved big job |
| `reserve_depth` | 1 | more blocked jobs get a protected start time |
| `fair_share_live` | `True` | consumption is recounted as the week goes on |

The three weights (<i style="color:#8a8fa3">w_age</i>, <i style="color:#8a8fa3">w_fair_share</i>, <i style="color:#8a8fa3">w_qos</i>) are the interesting ones. They are the whole policy.

## 6.2 · Reading the scorecard

Every `sv.evaluate` call adds a column of eight numbers, in four pairs: **performance**, **user
experience**, **infrastructure**, **equity**. Those exact words are the grey headers you are about
to see in the output.

Four of the eight you already know from Part 1, just averaged over forty jobs instead of read off
five by hand: **makespan**, **average waiting**, **average turnaround**, **utilisation**. Four are
new.

**Throughput** is jobs finished divided by makespan, in jobs per hour: more work cleared per hour
of wall-clock time.

**GPU utilisation** is utilisation again, counted only over the cluster's four GPUs. A schedule can
sit at 80 % busy on cores while every GPU is idle, if none of the running jobs happen to need one.

**Equity** is 100 % when every group received exactly the share it was promised in the first
24 hours, and falls as any group drifts from that promise, served too little or too much.

**Worst group's wait** is not everybody's average. It is the average wait of whichever one group
did worst, so a policy cannot hide one badly treated group behind three well treated ones.

Run the middle-of-the-road setting once, and read all eight before you touch a weight. This is
your reference point for everything after it.

In [58]:
sv.clear_leaderboard()
base, m = sv.evaluate(new_week, dict(w_age=0.5, w_fair_share=0.3, w_qos=0.2), name="balanced")

## 6.3 · 🎯 Take two positions

Now the work. Write **two** policies you could genuinely defend, and make them argue with each
other.

- **Policy A.** The head of the institute asks why the cluster is idle so often. Get the most work
  through it that you can.
- **Policy B.** The smallest group says it never gets a turn. Hold every group to the share it was
  promised.

Only the three weights change. Think about which one each policy leans on before you type.

In [59]:
fast, m_fast = sv.evaluate(new_week, dict(w_age=0.8, w_fair_share=0.0, w_qos=0.2),
                           name="A · throughput", quiet=True)

fair, m_fair = sv.evaluate(new_week, dict(w_age=0.2, w_fair_share=0.8, w_qos=0.0),
                           name="B · equity")

### What the scorecard should be showing you

Read the three columns across. **Policy A finishes the week five hours sooner and keeps about
eleven more points of the cluster busy. Policy B is the only one that comes close to the promised
shares.** Neither column wins the whole board, and no third column exists that wins both.

That is the finding, and it is the same one every cluster administrator meets: **the shares and the
throughput are bought from each other.** The scorecard cannot tell you which to buy. Only the
people who own the cluster can.

## 6.4 · Who paid for it

Totals hide people. The per group table is the one you will actually be asked about, so look at
what your two policies did to each group.

In [60]:
#@title 📊 What each group got, under A and under B (run me) { display-mode: "form" }
sv.group_card(fast, title="A · throughput: what each group got")
sv.group_card(fair, title="B · equity: what each group got")

group,target,delivered,avg wait
Biomedical Imaging9 jobs,40%,21.4% delivered · tick = target,4.1 h
Climate Modelling13 jobs,30%,48.2% delivered · tick = target,5.1 h
Robotics15 jobs,20%,26.8% delivered · tick = target,4.8 h
Economics3 jobs,10%,3.7% delivered · tick = target,7.0 h


group,target,delivered,avg wait
Biomedical Imaging9 jobs,40%,26.0% delivered · tick = target,2.2 h
Climate Modelling13 jobs,30%,38.7% delivered · tick = target,6.7 h
Robotics15 jobs,20%,24.9% delivered · tick = target,6.5 h
Economics3 jobs,10%,10.4% delivered · tick = target,1.0 h


The figures quoted below are for **A = 0.8 / 0.0 / 0.2** and **B = 0.2 / 0.8 / 0.0**. If you chose
differently your numbers will differ, but the shape of the story stays the same.

Three things are worth reading off those two tables.

**Economics is the group that Policy B rescues.** It was promised 10 % and Policy A gave it 3.7 %,
with the longest average wait of any group at 7 hours. Policy B gives it 10.4 % and its wait falls
to about 1 hour.

**Climate and Robotics pay the bill.** Both were being served ahead of their promised share, and
both now wait roughly an hour and a half longer. Nobody did anything wrong. The rule simply changed.

**Biomedical Imaging stays below its 40 % under both policies**, and that is the honest limit of
fair share: it only submitted 9 of the 40 jobs. A scheduler can stop a group being crowded out. It
cannot hand a group hours that the group never asked for.

## 6.5 · One number that lies

Backfilling is the setting whose value you can already predict. Switch it off and watch the equity
row, which is about to make an argument you should refuse.

In [61]:
no_bf, m_no_bf = sv.evaluate(new_week, dict(backfill=False), name="no backfilling")

### 🧠 Quick check

In [62]:
#@title 🧠 Quick check (run me) { display-mode: "form" }
sv.mc_quiz("equity_trap")

> ### What you just did
> You picked two policies and watched the same cluster serve two different weeks. One finished
> sooner and kept more of the cluster busy. The other kept the promises made to the four groups.
> Neither number is wrong, and no setting made both of them true at once. That choice is the job.

---
## In real life

This is the script you actually run to ask Slurm for the resources your job needs: how many
nodes, how much memory, how long you need them for.

In [63]:
#@title 📋 The sbatch script, field by field (run me) { display-mode: "form" }
sv.sbatch_card()

---
# Conclusion

Seven parts ago this was a cluster with two jobs running and five people waiting. Here is the same
walk again, backwards, in what each part actually left you with.

| part | | what you found out |
|:--:|---|---|
| **0** | *What fits* | free cores are not the same as usable cores. What is idle can still be the wrong shape. |
| **1** | *By hand* | the simplest rule anybody can write, FIFO, left 14 idle node-hours out of 48 without anybody making a mistake. |
| **2** | *Automate* | a scheduler is four lines in a loop: release, order the queue, fit what fits, advance the clock. |
| **3** | *Wall time* | a job is a promise about size and duration, never a measurement. Declare too little and it dies; too much and it stops fitting anywhere. |
| **4** | *Priority* | the weights you choose decide whose work matters. No priority rule is neutral. |
| **5** | *Backfilling* | one number, the wall time, decides whether a small job gets to fill a hole or waits for nothing. |
| **6** | *Your policy* | throughput against fairness, two policies that both defend gave two different weeks. Neither number was wrong. |

Put together, that is the whole job.

<div style="border-left:4px solid #764ba2;background:#f7f3ff;border-radius:0 10px 10px 0;padding:15px 19px;margin:16px 0;color:#24262b;font-size:15px;line-height:1.62"><div style="font-size:11px;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:#5b3a80;margin-bottom:8px">The one sentence to leave with</div><b>Slurm turns competing demands into allocations</b>, according to technical requirements and institutional policy. The one input entirely under your control is the rectangle you declare: how many nodes, how much memory, how long.</div>

Whatever gets built or picked apart in the rest of this course, it comes down to that rectangle,
and to who decided it should go first.